# Week 05/06 — Data Quality Checks

**Project:** RoadSafe — Collision Risk Analytics  
**Team:** Team 13  

This notebook performs data quality checks on the Silver Candidate layer before Gold-layer processing.

## 1. Load Silver Candidate Data

Update the table/path below if the Silver Candidate table in the project uses a different name.

In [ ]:
# Replace this with the actual Silver Candidate table name if required
silver_table = "silver_candidate"

silver_df = spark.table(silver_table)

print(f"Silver records loaded: {silver_df.count()}")
display(silver_df.limit(10))

## 2. Schema and Record Count Check

In [ ]:
silver_df.printSchema()

total_records = silver_df.count()
print(f"Total Silver records: {total_records}")

## 3. Null Value Check

In [ ]:
from pyspark.sql import functions as F

null_checks = []

for column in silver_df.columns:
    null_count = silver_df.filter(F.col(column).isNull()).count()
    null_checks.append((column, null_count))

null_df = spark.createDataFrame(
    null_checks,
    ["column_name", "null_count"]
)

display(null_df.orderBy(F.desc("null_count")))

## 4. Duplicate Record Check

In [ ]:
distinct_records = silver_df.dropDuplicates().count()
duplicate_records = total_records - distinct_records

print(f"Total records: {total_records}")
print(f"Distinct records: {distinct_records}")
print(f"Duplicate records: {duplicate_records}")

## 5. Negative Numeric Value Check

In [ ]:
numeric_columns = [
    column for column, dtype in silver_df.dtypes
    if dtype in ["int", "bigint", "double", "float"]
]

negative_checks = []

for column in numeric_columns:
    negative_count = silver_df.filter(F.col(column) < 0).count()
    negative_checks.append((column, negative_count))

negative_df = spark.createDataFrame(
    negative_checks,
    ["column_name", "negative_count"]
)

display(negative_df)

## 6. Data Quality Summary

In [ ]:
quality_summary = [
    ("Total Records", total_records),
    ("Distinct Records", distinct_records),
    ("Duplicate Records", duplicate_records)
]

summary_df = spark.createDataFrame(
    quality_summary,
    ["check", "value"]
)

display(summary_df)

## 7. Final Quality Gate

In [ ]:
quality_pass = (
    total_records > 0
    and duplicate_records == 0
)

if quality_pass:
    print("PASS: Silver data passed the basic quality checks.")
else:
    print("FAIL: Silver data requires further cleaning before Gold processing.")

## 8. Conclusion

The Silver Candidate dataset was checked for record availability, null values, duplicate records, and invalid negative numeric values. The quality gate indicates whether the dataset is ready to proceed toward the Gold layer.